# Modeling Benzene with a Model Hamiltonian and QPE Resource Estimation

This notebook demonstrates how to:
1. Build a **Pariser-Parr-Pople (PPP) model Hamiltonian** for benzene's π-system
2. Map it to a qubit Hamiltonian via Jordan-Wigner
3. Prepare a trial state and construct an **iterative QPE** circuit
4. Run **resource estimation** to understand the cost on fault-tolerant hardware

Benzene has 6 carbon atoms, each contributing one π-electron and one $p_z$ orbital.
This gives us a 6-site ring with 6 electrons — mapped to **12 qubits** (6 sites × 2 spins).

In [1]:
import numpy as np
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import (
    create_ppp_hamiltonian,
    ohno_potential,
)

# Reduce logging noise
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

## Step 1: Define the Benzene Lattice

Benzene's π-system is a 6-site periodic ring. Each site corresponds to one carbon $p_z$ orbital.
We use standard PPP parameters for benzene:
- **$t = 2.4$ eV** (hopping integral between adjacent carbons)
- **$U = 11.26$ eV** (on-site Coulomb repulsion, Ohno parameterization)
- **Bond length $R = 1.40$ Å** (C–C bond distance in benzene)

In [2]:
# Build a 6-site periodic ring (benzene π-system)
lattice = LatticeGraph.chain(n=6, periodic=True)
print(f"Lattice sites: {lattice.num_sites}")
print(f"Topology: 6-site periodic ring (benzene)")

# PPP parameters for benzene
t = 2.4          # hopping integral (eV)
U = 11.26        # on-site repulsion (eV)
bond_length = 1.40  # C-C bond length in Angstroms

# Compute nearest-neighbor repulsion using the Ohno potential
V = ohno_potential(lattice, U=U, R=bond_length, nearest_neighbor_only=True)
print(f"\nModel parameters:")
print(f"  t (hopping)     = {t:.2f} eV")
print(f"  U (on-site)     = {U:.2f} eV")
# V is a matrix; extract the nearest-neighbor value (site 0 to site 1)
V_nn = V[0, 1] if V.ndim == 2 else V[0]
print(f"  V (Ohno, n.n.)  = {V_nn:.4f} eV")
print(f"  U/t ratio       = {U/t:.2f} (strongly correlated regime)")

Lattice sites: 6
Topology: 6-site periodic ring (benzene)

Model parameters:
  t (hopping)     = 2.40 eV
  U (on-site)     = 11.26 eV
  V (Ohno, n.n.)  = 0.7129 eV
  U/t ratio       = 4.69 (strongly correlated regime)


In [3]:
from qdk.widgets import MoleculeViewer

# Visualize benzene geometry (approximate planar hexagonal structure)
# Since we're using a model Hamiltonian, we construct benzene coordinates manually
benzene_xyz = """12
Benzene (C6H6) - model geometry
C    1.2124    0.7000    0.0000
C    1.2124   -0.7000    0.0000
C    0.0000   -1.4000    0.0000
C   -1.2124   -0.7000    0.0000
C   -1.2124    0.7000    0.0000
C    0.0000    1.4000    0.0000
H    2.1562    1.2450    0.0000
H    2.1562   -1.2450    0.0000
H    0.0000   -2.4900    0.0000
H   -2.1562   -1.2450    0.0000
H   -2.1562    1.2450    0.0000
H    0.0000    2.4900    0.0000
"""

MoleculeViewer(molecule_data=benzene_xyz)

## Step 2: Build the Fermionic Hamiltonian

We construct a PPP (Pariser-Parr-Pople) Hamiltonian, which is an extended Hubbard model with long-range Coulomb interactions parameterized by the Ohno potential. This captures the essential π-electron physics of benzene.

In [4]:
# Build the PPP Hamiltonian for benzene
hamiltonian = create_ppp_hamiltonian(lattice, epsilon=0.0, t=t, U=U, V=V, z=1.0)
print("Fermionic Hamiltonian summary:")
print(hamiltonian.get_summary())

Fermionic Hamiltonian summary:
Hamiltonian Summary:
  Type: Hermitian
  Restrictedness: Restricted
  Active Orbitals: 6
  Total Orbitals: 6
  Core Energy: 4.277117
  Integral Statistics:
    One-body Integrals (alpha): 36 (larger than 0.000001: 18)
    Two-body Integrals (aaaa): 1296 (larger than 0.000001: 18)



## Step 3: Classical Reference (Exact Diagonalization)

Before running QPE, we compute the exact ground-state energy classically using CASCI. This gives us a reference to compare the QPE result against.
With 6 electrons in 6 orbitals, this is still tractable classically — but the model serves as a prototype for larger systems where classical methods fail.

In [5]:
# Solve exactly with CASCI (6 electrons: 3 alpha, 3 beta in 6 orbitals)
n_alpha, n_beta = 3, 3
mc = create("multi_configuration_calculator", "macis_cas")
e_exact, wfn_exact = mc.run(hamiltonian, n_alpha, n_beta)

print(f"Exact CASCI ground-state energy: {e_exact:.6f} eV")
print(f"Number of electrons: {n_alpha + n_beta} ({n_alpha}α, {n_beta}β)")
print(f"Number of orbitals (sites): 6")
print(f"\nTop determinants in the ground state:")
top_dets = wfn_exact.get_top_determinants(5)
for det, coeff in top_dets.items():
    print(f"  {det}: {coeff:.4f}")

Exact CASCI ground-state energy: -8.156231 eV
Number of electrons: 6 (3α, 3β)
Number of orbitals (sites): 6

Top determinants in the ground state:
  ududud00: 0.3390
  dududu00: 0.3390
  ududdu00: 0.1482
  uuddud00: 0.1482
  uduudd00: 0.1482


## Step 4: Jordan-Wigner Qubit Mapping

We map the fermionic Hamiltonian to qubit operators. The Jordan-Wigner encoding maps each spin-orbital to one qubit: 6 sites × 2 spins = **12 qubits**.

In [6]:
# Map fermionic Hamiltonian to qubit operators (Jordan-Wigner)
qubit_mapper = create("qubit_mapper", algorithm_name="qdk", encoding="jordan-wigner")
qubit_hamiltonian = qubit_mapper.run(hamiltonian)

print("Qubit Hamiltonian summary:")
print(qubit_hamiltonian.get_summary())
print(f"\nNumber of qubits: {qubit_hamiltonian.num_qubits}")
print(f"Number of Pauli terms: {len(qubit_hamiltonian.pauli_strings)}")
print(f"Schatten (L1) norm: {qubit_hamiltonian.schatten_norm:.4f}")

Qubit Hamiltonian summary:
Qubit Hamiltonian
  Number of qubits: 12
  Number of terms: 67
  Encoding: jordan-wigner
  Fermion mode order: blocked


Number of qubits: 12
Number of Pauli terms: 67
Schatten (L1) norm: 96.3600


## Step 5: State Preparation

We prepare a trial state for QPE using the top determinants from the exact solution. In practice, you'd use a cheaper classical method (like Hartree-Fock) to get an initial guess — here we use the exact state for a clean demonstration.

In [7]:
from qdk_chemistry.data import Wavefunction, CasWavefunctionContainer

# Build a trial state from the top 6 determinants
dets = wfn_exact.get_top_determinants(6)
total_weight = sum(np.abs(c)**2 for c in dets.values())
dets = {det: c / np.sqrt(total_weight) for det, c in dets.items()}

det_keys = list(dets.keys())
coeffs = np.array(list(dets.values()))
trial_wfn = Wavefunction(CasWavefunctionContainer(coeffs, det_keys, wfn_exact.get_orbitals()))

# Generate state preparation circuit
state_prep = create("state_prep", "sparse_isometry_gf2x")
state_prep_circuit = state_prep.run(trial_wfn)
print(f"State preparation circuit generated")

State preparation circuit generated


## Step 6: Construct the iQPE Circuit

We build an iterative QPE circuit with:
- **8-bit precision** for the phase readout
- **2nd-order Trotter** decomposition for time evolution
- Evolution time set by $t = \pi / \|H\|_1$ (Schatten norm bound)

In [8]:
# Configure iQPE
M_PRECISION = 8  # bits of phase precision
evolution_time = np.pi / qubit_hamiltonian.schatten_norm
print(f"Evolution time: {evolution_time:.6f} (eV^-1)")

# Create algorithm components
evolution_builder = create("time_evolution_builder", "trotter", order=2)
circuit_mapper = create("controlled_evolution_circuit_mapper", "pauli_sequence")
iqpe = create(
    "phase_estimation", "iterative",
    num_bits=M_PRECISION,
    evolution_time=evolution_time,
    shots_per_bit=3,
)

# Generate the most expensive iQPE iteration circuit (iteration 0 = highest power of U)
iqpe_circuit = iqpe.create_iteration_circuit(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
    iteration=0,
    total_iterations=M_PRECISION,
)
print(f"iQPE circuit constructed (most expensive iteration)")

Evolution time: 0.032603 (eV^-1)
iQPE circuit constructed (most expensive iteration)


In [9]:
from qdk.widgets import Circuit

# Visualize the state preparation circuit
print("State Preparation Circuit:")
display(Circuit(state_prep_circuit.get_qsharp_circuit()))

# Visualize one iteration of the iQPE circuit
print("\niQPE Iteration Circuit (most expensive iteration, 2^7 Trotter steps):")
display(Circuit(iqpe_circuit.get_qsharp_circuit()))

State Preparation Circuit:



iQPE Iteration Circuit (most expensive iteration, 2^7 Trotter steps):


## Step 7: Resource Estimation

Now we use the **Azure Quantum Resource Estimator** to determine the physical resources needed to run this QPE circuit on fault-tolerant hardware. We estimate:
- Physical qubits required
- Algorithmic (logical) qubits
- T-gate count
- Runtime

We use gate-based superconducting qubits with surface code error correction as the target architecture.

In [10]:
from qdk.estimator import EstimatorParams, QubitParams, QECScheme

# Configure resource estimation parameters
params = EstimatorParams()
params.error_budget = 0.01  # 1% total error budget
params.qubit_params.name = QubitParams.GATE_NS_E3  # superconducting, gate-based, 10^-3 error rate
params.qec_scheme.name = QECScheme.SURFACE_CODE     # surface code QEC

# Run resource estimation on the iQPE circuit
result = iqpe_circuit.estimate(params)

# Extract and display results
physical = result["physicalCounts"]
breakdown = physical["breakdown"]

print("=" * 60)
print("RESOURCE ESTIMATION: Benzene PPP Model — iQPE (8-bit)")
print("=" * 60)
print(f"\n{'Physical qubits:':<35} {physical['physicalQubits']:,}")
print(f"{'Algorithmic logical qubits:':<35} {breakdown['algorithmicLogicalQubits']}")
print(f"{'T-gate count:':<35} {breakdown.get('numTstates', 'N/A'):,}")
print(f"{'Runtime:':<35} {physical['runtime']}")
print(f"\n{'Error budget:':<35} {params.error_budget}")
print(f"{'QEC scheme:':<35} Surface Code")
print(f"{'Qubit model:':<35} gate_ns_e3 (superconducting)")
print("=" * 60)

RESOURCE ESTIMATION: Benzene PPP Model — iQPE (8-bit)

Physical qubits:                    363,964
Algorithmic logical qubits:         38
T-gate count:                       608,814
Runtime:                            2990735200

Error budget:                       0.01
QEC scheme:                         Surface Code
Qubit model:                        gate_ns_e3 (superconducting)


/tmp/ipykernel_486417/1432624248.py:4: DeprecationWarning: This version of QRE is deprecated and will be removed in a future release. Please use the new version of QRE in qdk.qre. Refer to aka.ms/qdk.QREv3 for more information.
  params = EstimatorParams()


In [11]:
from qdk.widgets import EstimateDetails, SpaceChart

# Interactive resource estimation details table
print("Detailed Resource Breakdown:")
display(EstimateDetails(result))

# Physical qubit space distribution chart
print("\nPhysical Qubit Distribution:")
display(SpaceChart(result))

Detailed Resource Breakdown:



Physical Qubit Distribution:


## Step 8: Run QPE Simulation

Finally, we run the full iQPE algorithm on the QDK simulator to verify it recovers the correct ground-state energy.

In [12]:
# Run iQPE on the QDK simulator
circuit_executor = create("circuit_executor", "qdk_sparse_state_simulator")
qpe_result = iqpe.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    circuit_executor=circuit_executor,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
)

# Compare with exact result
estimated_energy = qpe_result.raw_energy + hamiltonian.get_core_energy()
error = abs(estimated_energy - e_exact)

print(f"{'Exact CASCI energy:':<35} {e_exact:.6f} eV")
print(f"{'QPE estimated energy:':<35} {estimated_energy:.6f} eV")
print(f"{'Absolute error:':<35} {error:.3e} eV")
print(f"{'Phase precision:':<35} {M_PRECISION} bits")

Exact CASCI energy:                 -8.156231 eV
QPE estimated energy:               -26.588195 eV
Absolute error:                     1.843e+01 eV
Phase precision:                    8 bits
